# ScholarLens 03 - Scientific NER (SciBERT vs BERT-base)
Owner: Member 3. SciERC download + BIO conversion -> dictionary baseline -> fine-tune both models -> per-type analysis.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sajithaucm/Scholarlens_Project/blob/main/notebooks/03_ner.ipynb)

## 0. Setup (run every time you open this notebook)
1. **Runtime -> Change runtime type -> T4 GPU** (Hardware accelerator).
2. Replace `sajithaucm` below with your team's GitHub account (once, then commit).
3. Checkpoints are saved to **Google Drive** so they survive a disconnect; small result files are saved in the repo's `results/` folder and pushed to GitHub at the end.

In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

REPO = "https://github.com/sajithaucm/Scholarlens_Project.git"
if not os.path.exists("/content/Scholarlens_Project"):
    !git clone $REPO /content/Scholarlens_Project
%cd /content/Scholarlens_Project
!git pull -q
!pip install -q -r requirements.txt

# Checkpoints -> Google Drive (big files never go to GitHub)
os.environ["SCHOLARLENS_OUTPUT_DIR"] = "/content/drive/MyDrive/Scholarlens_Project/outputs"
os.makedirs(os.environ["SCHOLARLENS_OUTPUT_DIR"], exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip freeze > results/environment.txt   # exact library versions, for reproducibility

## 1. Data check (downloads SciERC from the authors' site and converts to BIO)

In [ ]:
!python -m src.data_loading --task ner

## 2. Quick test

In [ ]:
!python -m src.ner.train --model bert-base-uncased --max-samples 200 --epochs 1

## 3. Dictionary baseline

In [ ]:
!python -m src.baselines --task ner

## 4. Fine-tune SciBERT

In [ ]:
!python -m src.ner.train --model allenai/scibert_scivocab_uncased

## 5. Fine-tune BERT-base

In [ ]:
!python -m src.ner.train --model bert-base-uncased

## 6. Per-type F1 and errors (which types are confused? boundary errors?)

In [ ]:
import json, pandas as pd
per = {}
for m in ('allenai__scibert_scivocab_uncased', 'bert-base-uncased', 'baseline-dictionary'):
    per[m] = json.load(open(f'results/ner_{m}.json'))['metrics']['per_type_f1']
pd.DataFrame(per)

In [ ]:
rows = [json.loads(l) for l in open('results/ner_allenai__scibert_scivocab_uncased_samples.jsonl')]
for r in [r for r in rows if not r['all_correct']][:5]:
    print(' '.join(f'{t}[{g}|{p}]' if g != p else t for t, g, p in zip(r['tokens'], r['gold'], r['pred'])), '\n')

In [ ]:
!python -m src.compare

## Save results to GitHub
Results (`results/*.json`, figures, sample predictions) are small and must be committed.
Create a GitHub **fine-grained personal access token** (repo contents: read & write) and paste it when asked. It is not stored in the notebook.
Alternative: download the `results/` folder (Files panel) and commit it from VS Code.

In [ ]:
from getpass import getpass
name, email = input("Your name: "), input("Your GitHub email: ")
token = getpass("GitHub token: ")
!git config user.name "$name" && git config user.email "$email"
!git add results/
!git commit -m "Add results from Colab run" || echo "nothing to commit"
!git push https://$token@github.com/sajithaucm/Scholarlens_Project.git HEAD:main
del token